# Контрольная работа №1. Проектирование и построение базы знаний
## Выполнение заданий 5 и 6: Текстовый корпус, проверка и логический вывод

**Предметная область:** Программные библиотеки глубокого обучения экосистемы PyTorch  
**Основной класс в Wikidata:** `wd:Q29642950` Python library, связанный отношением `wdt:P1547 wd:Q47509047` зависит от PyTorch.

Работа продолжает [задания 3–4](задания_3-4.ipynb) и использует тот же сохранённый снимок данных.
Условие: [Контрольная работа 1.docx](../Контрольная%20работа%201.docx).
Все результаты ниже получаются при последовательном выполнении ячеек.

Для запуска нужны `rdflib`, `pyshacl`, `owlrl`, `pandas` и `ipykernel`:
`python -m pip install rdflib pyshacl owlrl pandas ipykernel`.
Ноутбук запускается из корня проекта или каталога `Резы`; доступ к сети для воспроизведения не требуется.


In [1]:
import hashlib
import json
import platform
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import pyshacl
import owlrl
from IPython.display import display
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal, URIRef
from rdflib.compare import isomorphic

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

project = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'knowledge_graph.ttl').is_file() and (p / 'data/corpus.jsonl').is_file()
)
artifacts = project / 'data' / 'tasks_5_6'
artifacts.mkdir(parents=True, exist_ok=True)
PTE = Namespace('http://example.org/pytorch-ecosystem#')
WD = Namespace('http://www.wikidata.org/entity/')
WDT = Namespace('http://www.wikidata.org/prop/direct/')
SH = Namespace('http://www.w3.org/ns/shacl#')

environment = {'Python': platform.python_version()}
environment.update({name: version(name) for name in ['rdflib', 'pyshacl', 'owlrl', 'pandas']})
display(pd.DataFrame(environment.items(), columns=['Компонент', 'Версия']))
(artifacts / 'environment.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)


,Компонент,Версия
0,Python,3.12.15
1,rdflib,7.6.0
2,pyshacl,0.40.2
3,owlrl,7.6.2
4,pandas,3.1.1


110

<p class="task" id="5"></p>

## Задание 5. Сбор текстовой информации по сущностям

### 5.1. Источники и структура корпуса

Список сущностей берётся из `data/task1_entities.json`: это 52 библиотеки, выбранные в задании 1.
Вспомогательные узлы графа — лицензии, репозитории, релизы и центральный узел PyTorch — не являются дополнительными сущностями этой выборки.

Для каждой библиотеки используется README из её репозитория GitHub. Скрипт
`scripts/01_fetch_data_and_corpus.py` ранее сохранил первые 4 000 символов README в `data/corpus.jsonl`.
В этой работе корпус собирается из этих локальных выдержек: повторное скачивание могло бы изменить тексты и цитаты из задания 2.
Это **повторная сборка сохранённого снимка**, а не новая загрузка страниц.
Исходный скрипт допускает шаблонный текст при ошибке скачивания; ниже отдельно проверяется, что таких подстановок в используемом корпусе нет.

Формат результата — UTF-8 JSONL: один JSON-объект на строку, переводы строк внутри текста экранированы.
Разметка Markdown/RST сохранена, чтобы не менять исходные цитаты. Результат записывается в
[`data/tasks_5_6/corpus.jsonl`](../data/tasks_5_6/corpus.jsonl).

| Поле | Содержание |
| :--- | :--- |
| `id`, `name` | QID Wikidata и название библиотеки |
| `source_url` | Адрес исходного файла README; один источник на запись |
| `source_type` | `github_readme` |
| `description` | Описание из сохранённой выгрузки Wikidata |
| `text` | Сохранённая выдержка README, не более 4 000 символов |
| `text_sha256` | Контрольная сумма текста для проверки воспроизводимости |
| `provenance` | Путь к локальному снимку, из которого взят текст |

Дата первоначального скачивания и commit SHA в исходном корпусе не записаны. Поэтому ссылки указывают
на изменяемые ветки репозиториев; воспроизводимость обеспечивается локальным текстом и его контрольной суммой.


In [2]:
entities = json.loads((project / 'data/task1_entities.json').read_text(encoding='utf-8'))
original_corpus = [
    json.loads(line)
    for line in (project / 'data/corpus.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
entity_ids = {entity['qid'] for entity in entities}
source_ids = [doc['id'] for doc in original_corpus]
assert len(entity_ids) == len(entities) == 52
assert len(source_ids) == len(set(source_ids)), 'В исходном корпусе есть повторяющиеся QID'
assert set(source_ids) == entity_ids, 'Состав корпуса отличается от выборки задания 1'

corpus = []
for doc in sorted(original_corpus, key=lambda d: (d['name'].casefold(), d['id'])):
    source = urlparse(doc['source_url'])
    assert source.scheme == 'https' and source.netloc == 'raw.githubusercontent.com'
    assert source.path.lower().endswith(('readme.md', 'readme.rst'))
    assert isinstance(doc['text'], str) and doc['text'].strip()
    assert len(doc['text']) <= 4000
    assert 'is an open-source Python library designed for machine learning workflows' not in doc['text']
    corpus.append({
        **doc,
        'source_type': 'github_readme',
        'text_sha256': hashlib.sha256(doc['text'].encode('utf-8')).hexdigest(),
        'provenance': 'data/corpus.jsonl',
    })

corpus_path = artifacts / 'corpus.jsonl'
with corpus_path.open('w', encoding='utf-8', newline='\n') as file:
    for doc in corpus:
        file.write(json.dumps(doc, ensure_ascii=False) + '\n')

# Обратное чтение проверяет именно сохранённый JSONL, а не только список в памяти.
loaded_corpus = [json.loads(line) for line in corpus_path.read_text(encoding='utf-8').splitlines()]
assert loaded_corpus == corpus
assert {doc['id'] for doc in loaded_corpus} == entity_ids
for doc in loaded_corpus:
    assert hashlib.sha256(doc['text'].encode('utf-8')).hexdigest() == doc['text_sha256']

lengths = [len(doc['text']) for doc in loaded_corpus]
corpus_stats = {
    'Сущностей в выборке': len(entity_ids),
    'Документов / строк JSONL': len(loaded_corpus),
    'Сущностей с текстом и источником': len({doc['id'] for doc in loaded_corpus}),
    'Охват выборки, %': round(len(loaded_corpus) / len(entity_ids) * 100, 2),
    'Уникальных URL источников': len({doc['source_url'] for doc in loaded_corpus}),
    'Минимальная длина текста': min(lengths),
    'Максимальная длина текста': max(lengths),
    'Средняя длина текста': round(sum(lengths) / len(lengths), 2),
    'Размер JSONL, байт': corpus_path.stat().st_size,
}
display(pd.DataFrame(corpus_stats.items(), columns=['Метрика', 'Значение']))
print('Сохранено:', corpus_path.relative_to(project).as_posix())
print('Обратное чтение JSONL, покрытие QID и контрольные суммы: OK')


,Метрика,Значение
0,Сущностей в выборке,52.00
1,Документов / строк JSONL,52.00
2,Сущностей с текстом и источником,52.00
3,"Охват выборки, %",100.00
4,Уникальных URL источников,51.00
5,Минимальная длина текста,2069.00
6,Максимальная длина текста,4000.00
7,Средняя длина текста,3885.92
8,"Размер JSONL, байт",224584.00


Сохранено: data/tasks_5_6/corpus.jsonl
Обратное чтение JSONL, покрытие QID и контрольные суммы: OK


### 5.2. Источник для каждой сущности

Таблица содержит все 52 сущности, а не только примеры. Один README может соответствовать нескольким
QID: `pytorch-pretrained-bert` и `pytorch-transformers` используют общий репозиторий Transformers.
Требование проверяется по QID: у каждой сущности есть непустой текст и ссылка на источник.


In [3]:
sources_df = pd.DataFrame([{
    'QID': doc['id'],
    'Библиотека': doc['name'],
    'Источник': doc['source_url'],
    'Символов': len(doc['text']),
} for doc in loaded_corpus])
display(sources_df)
sources_df.to_csv(artifacts / 'corpus_sources.csv', index=False, encoding='utf-8-sig')

print('Пример записи JSONL (текст сокращён только при выводе):')
example = next(doc for doc in loaded_corpus if doc['name'] == 'torchvision')
print(json.dumps({**example, 'text': example['text'][:800] + '…'}, ensure_ascii=False, indent=2))


,QID,Библиотека,Источник,Символов
0,Q116908792,accelerate,https://raw.githubusercontent.com/huggingface/accelerate/master/README.md,4000
1,Q107382302,allennlp,https://raw.githubusercontent.com/allenai/allennlp/master/README.md,4000
2,Q107382376,attacut,https://raw.githubusercontent.com/PyThaiNLP/attacut/master/README.md,3419
3,Q107382224,bert-score,https://raw.githubusercontent.com/Tiiiger/bert_score/master/README.md,4000
4,Q107382330,botorch,https://raw.githubusercontent.com/meta-pytorch/botorch/master/README.md,4000
5,Q107382311,captum,https://raw.githubusercontent.com/meta-pytorch/captum/master/README.md,4000
6,Q107382139,catalyst,https://raw.githubusercontent.com/catalyst-team/catalyst/master/README.md,4000
7,Q120549792,clean-fid,https://raw.githubusercontent.com/GaParmar/clean-fid/master/README.md,4000
8,Q120596958,effdet,https://raw.githubusercontent.com/rwightman/efficientdet-pytorch/master/README.md,4000
9,Q120550306,face-alignment,https://raw.githubusercontent.com/1adrianb/face-alignment/master/README.md,4000


Пример записи JSONL (текст сокращён только при выводе):
{
  "id": "Q107381670",
  "name": "torchvision",
  "source_url": "https://raw.githubusercontent.com/pytorch/vision/master/README.md",
  "description": "Python library",
  "text": "# torchvision\n\n[![total torchvision downloads](https://pepy.tech/badge/torchvision)](https://pepy.tech/project/torchvision)\n[![documentation](https://img.shields.io/badge/dynamic/json.svg?label=docs&url=https%3A%2F%2Fpypi.org%2Fpypi%2Ftorchvision%2Fjson&query=%24.info.version&colorB=brightgreen&prefix=v)](https://pytorch.org/vision/stable/index.html)\n\nThe torchvision package consists of popular datasets, model architectures, and common image transformations for computer\nvision.\n\n## Installation\n\nPlease refer to the [official\ninstructions](https://pytorch.org/get-started/locally/) to install the stable\nversions of `torch` and `torchvision` on your system.\n\nTo build source, refer to our [contributing\npage](https://github.com/pytorch/vision/b

<p class="task" id="6"></p>

## Задание 6. Проверка данных и логический вывод

### 6.1. Ограничения SHACL

Проверяются **пять разных видов ограничений**, что превышает требование «не менее трёх».
Формы берутся из `shapes.shacl.ttl`. В рабочую копию добавлена явная цель `pte:AudioProcessingLibrary`,
чтобы в перечне целей были представлены все категории библиотек из задания 3.
Сами ограничения не ослабляются. Рабочая копия сохраняется в `data/tasks_5_6/shapes.shacl.ttl`.

| Вид | Ограничение | Что проверяется |
| :--- | :--- | :--- |
| Кардинальность | `sh:minCount`, `sh:maxCount` | Ровно одно имя и основной репозиторий; хотя бы одна лицензия; ровно один URL репозитория; хотя бы один идентификатор версии |
| Тип данных | `sh:datatype` | Имя и идентификатор версии — `xsd:string`; число версий — `xsd:integer`; URL — `xsd:anyURI`; дата — `xsd:dateTime` |
| Класс значения | `sh:class` | Значение `hasLicense` принадлежит `pte:SoftwareLicense` |
| Диапазон | `sh:minInclusive 0` | Число версий неотрицательно |
| Шаблон | `sh:pattern` | URL соответствует `https://github.com/owner/repo` с допустимым завершающим `/` |

Для даты и числа версий проверяются имеющиеся значения: обязательность их наличия формами не задана.
Шаблон URL проверяет структуру строки, но не доступность репозитория в сети.

Проверка выполняется **до OWL-вывода**, с `inference='none'`. Онтология передаётся отдельно в `ont_graph`,
чтобы SHACL учитывал иерархию классов. Предварительное применение `rdfs:range` могло бы автоматически
приписать ошибочному объекту класс лицензии и скрыть нарушение `sh:class`.


In [4]:
kg = Graph().parse(project / 'knowledge_graph.ttl', format='turtle')
ontology = Graph().parse(project / 'ontology.ttl', format='turtle')
shapes = Graph().parse(project / 'shapes.shacl.ttl', format='turtle')
shapes.add((PTE.LibraryShape, SH.targetClass, PTE.AudioProcessingLibrary))
shapes.serialize(destination=artifacts / 'shapes.shacl.ttl', format='turtle')
assert isomorphic(shapes, Graph().parse(artifacts / 'shapes.shacl.ttl', format='turtle'))

def clone_graph(graph):
    result = Graph()
    for prefix, namespace in graph.namespaces():
        result.bind(prefix, namespace)
    for triple in graph:
        result.add(triple)
    return result

def short(term):
    return term.n3(kg.namespace_manager) if term is not None else '—'

def validate_and_save(graph, name):
    conforms, report_graph, report_text = pyshacl.validate(
        data_graph=graph,
        shacl_graph=shapes,
        ont_graph=ontology,
        inference='none',
        abort_on_first=False,
        meta_shacl=True,
        advanced=False,
        inplace=False,
    )
    (artifacts / f'{name}.txt').write_text(report_text, encoding='utf-8')
    report_graph.serialize(destination=artifacts / f'{name}.ttl', format='turtle')
    return conforms, report_graph, report_text

def report_rows(report):
    rows = []
    for result in report.subjects(RDF.type, SH.ValidationResult):
        rows.append({
            'Узел': short(report.value(result, SH.focusNode)),
            'Свойство': short(report.value(result, SH.resultPath)),
            'Компонент': str(report.value(result, SH.sourceConstraintComponent)).split('#')[-1],
            'Значение': short(report.value(result, SH.value)),
            'Сообщение': str(report.value(result, SH.resultMessage)),
        })
    return sorted(rows, key=lambda row: (row['Компонент'], row['Узел'], row['Свойство']))

# Проверяем, что все библиотеки, включая аудио и центральный PyTorch, входят в область проверки.
library_targets = set(shapes.objects(PTE.LibraryShape, SH.targetClass))
targeted_libraries = {s for cls in library_targets for s in kg.subjects(RDF.type, cls)}
all_libraries = set(kg.subjects(PTE.libraryName, None))
assert targeted_libraries == all_libraries
graph_qids = {
    str(wd).rsplit('/', 1)[-1]
    for library in all_libraries for wd in kg.objects(library, OWL.sameAs)
}
assert entity_ids <= graph_qids, 'Не все сущности корпуса представлены в графе'
print('Троек данных:', len(kg))
print('Троек онтологии:', len(ontology))
print('Библиотек, охваченных формой:', len(targeted_libraries))
print('Репозиториев:', len(set(kg.subjects(RDF.type, PTE.Repository))))
print('Версий:', len(set(kg.subjects(RDF.type, PTE.SoftwareVersion))))


Троек данных: 9302
Троек онтологии: 155
Библиотек, охваченных формой: 53
Репозиториев: 53
Версий: 1719


### 6.2. Результат pyshacl на исходных данных

Ниже выводится полный текст отчёта pyshacl. Его текстовая и RDF-версии сохраняются рядом с ноутбуком
в каталоге результатов. Отсутствие нарушений означает соответствие **заданным формам**,
а не доказательство фактической корректности всех сведений из Wikidata.


In [5]:
kg_before_validation = set(kg)
base_conforms, base_report, base_text = validate_and_save(kg, 'shacl_original')
print(base_text)
print('Число нарушений:', len(report_rows(base_report)))
assert base_conforms and not report_rows(base_report)
assert set(kg) == kg_before_validation, 'Проверка изменила исходный граф'


Validation Report
Conforms: True

Число нарушений: 0


### 6.3. Контрольный эксперимент: выявление пяти типов нарушений

В исходном снимке нарушений выбранных ограничений не обнаружено: данные и ограничения оставлены без исправлений.
Чтобы показать работу диагностики и процедуру исправления, создаётся **отдельная копия графа с искусственными ошибками**.
Эти ошибки не выдаются за обнаруженные дефекты исходного набора. В копию вносятся:

1. Второе название библиотеки — нарушение максимальной кардинальности.
2. Число вместо строкового идентификатора версии — нарушение типа данных.
3. Ссылка `hasLicense` на репозиторий — нарушение класса значения.
4. Отрицательное число версий — нарушение диапазона.
5. URL с `http` вместо `https` — нарушение шаблона при сохранённом типе `xsd:anyURI`.

Изменения подобраны так, чтобы каждая ошибка нарушала один соответствующий вид ограничения.
Выполняется тот же набор форм и тот же режим проверки.


In [6]:
demo = clone_graph(kg)
library = PTE.Lib_PyKEEN_Q96631404
repository = kg.value(library, PTE.hasSourceCodeRepository)
release = sorted(kg.objects(library, PTE.hasVersion), key=str)[0]
assert repository is not None

patches = [
    (library, PTE.libraryName, Literal('Второе тестовое имя', datatype=XSD.string), False),
    (release, PTE.versionIdentifier, Literal(123, datatype=XSD.integer), True),
    (library, PTE.hasLicense, repository, True),
    (library, PTE.versionCount, Literal(-1, datatype=XSD.integer), True),
    (repository, PTE.repositoryURL, Literal('http://github.com/pykeen/pykeen', datatype=XSD.anyURI), True),
]
original_values = {}
for subject, predicate, bad_value, replace in patches:
    original_values[(subject, predicate)] = list(kg.objects(subject, predicate))
    if replace:
        demo.remove((subject, predicate, None))
    demo.add((subject, predicate, bad_value))

demo.serialize(destination=artifacts / 'validation_demo_invalid.ttl', format='turtle')
demo_conforms, demo_report, demo_text = validate_and_save(demo, 'shacl_demo_before')
print(demo_text)
violations = report_rows(demo_report)
display(pd.DataFrame(violations))
expected_components = {
    'MaxCountConstraintComponent', 'DatatypeConstraintComponent',
    'ClassConstraintComponent', 'MinInclusiveConstraintComponent', 'PatternConstraintComponent',
}
assert not demo_conforms
assert len(violations) == 5
assert {row['Компонент'] for row in violations} == expected_components


Validation Report
Conforms: False
Results (5):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class pte:SoftwareLicense ; sh:message Literal("Лицензия библиотеки должна быть объектом класса SoftwareLicense.", lang=ru) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path pte:hasLicense ]
	Focus Node: pte:Lib_PyKEEN_Q96631404
	Value Node: pte:Repo_PyKEEN_Q96631404
	Result Path: pte:hasLicense
	Message: Лицензия библиотеки должна быть объектом класса SoftwareLicense.
Constraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:message Literal("Идентификатор версии должен быть строкой.", lang=ru) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path pte:versionIdentifier ]
	Focus Node: pte:Ver_PyKEEN_Q96631404_0_0_26_18
	Value Node: Literal("123", datatype=xsd:int

,Узел,Свойство,Компонент,Значение,Сообщение
0,pte:Lib_PyKEEN_Q96631404,pte:hasLicense,ClassConstraintComponent,pte:Repo_PyKEEN_Q96631404,Лицензия библиотеки должна быть объектом класса SoftwareLicense.
1,pte:Ver_PyKEEN_Q96631404_0_0_26_18,pte:versionIdentifier,DatatypeConstraintComponent,"""123""^^xsd:integer",Идентификатор версии должен быть строкой.
2,pte:Lib_PyKEEN_Q96631404,pte:libraryName,MaxCountConstraintComponent,—,Библиотека должна иметь ровно одно строковое название (libraryName).
3,pte:Lib_PyKEEN_Q96631404,pte:versionCount,MinInclusiveConstraintComponent,"""-1""^^xsd:integer",Число версий должно быть неотрицательным целым числом.
4,pte:Repo_PyKEEN_Q96631404,pte:repositoryURL,PatternConstraintComponent,"""http://github.com/pykeen/pykeen""^^xsd:anyURI",URL репозитория должен указывать на валидный адрес GitHub репозитория.


### 6.4. Решения по каждому типу нарушения и повторная проверка

| Вид нарушения в эксперименте | Решение | Что сделано и почему |
| :--- | :--- | :--- |
| Кардинальность | Данные исправлены | Удалено добавленное тестовое имя, восстановлено единственное исходное название. В модели задано одно основное имя. |
| Тип данных | Данные исправлены | Восстановлен исходный строковый идентификатор версии. Преобразование числа `123` в строку не восстановило бы правильный номер релиза. |
| Класс значения | Данные исправлены | Восстановлена исходная ссылка на узел лицензии. Репозиторий не объявляется лицензией только ради прохождения проверки. |
| Диапазон | Данные исправлены | Восстановлено исходное неотрицательное число версий. Отрицательное число не имеет смысла для счётчика. |
| Шаблон | Данные исправлены | Восстановлен исходный HTTPS-адрес репозитория с типом `xsd:anyURI`. Формат URL согласован с сохранёнными данными. |

Ограничения не ослаблены; нарушений в исправленной копии не оставлено. Восстановление выполняется
по значениям исходного графа, сохранённым **до** эксперимента. Для реальных данных выбор нового значения
потребовал бы сверки с источником. Сравнение множеств троек проверяет, что исправленная копия полностью совпала с исходной.


In [7]:
repaired = clone_graph(demo)
for (subject, predicate), values in original_values.items():
    repaired.remove((subject, predicate, None))
    for value in values:
        repaired.add((subject, predicate, value))
assert set(repaired) == set(kg)

fixed_conforms, fixed_report, fixed_text = validate_and_save(repaired, 'shacl_demo_after')
print(fixed_text)
assert fixed_conforms and not report_rows(fixed_report)
validation_summary = pd.DataFrame([
    {'Набор': 'Исходный граф', 'Conforms': base_conforms, 'Нарушений': len(report_rows(base_report))},
    {'Набор': 'Копия с пятью искусственными ошибками', 'Conforms': demo_conforms, 'Нарушений': len(violations)},
    {'Набор': 'Исправленная копия', 'Conforms': fixed_conforms, 'Нарушений': len(report_rows(fixed_report))},
])
display(validation_summary)
validation_summary.to_csv(artifacts / 'validation_summary.csv', index=False, encoding='utf-8-sig')


Validation Report
Conforms: True



,Набор,Conforms,Нарушений
0,Исходный граф,True,0
1,Копия с пятью искусственными ошибками,False,5
2,Исправленная копия,True,0


### 6.5. Логический вывод OWL 2 RL

Для вывода берётся объединение **исходного графа данных и онтологии**, без SHACL-форм и экспериментальных ошибок.
`owlrl.DeductiveClosure(owlrl.OWLRL_Semantics)` достраивает замыкание по правилам OWL 2 RL.
Перед запуском фиксируется множество утверждённых троек; новые тройки вычисляются как точная разность
`после − до`. Поэтому уже существовавшие связи не попадают в список добавленных.

Полная разность сохраняется в [`owlrl_added_triples.ttl`](../data/tasks_5_6/owlrl_added_triples.ttl).
Она содержит как предметные выводы, так и служебные утверждения о классах, свойствах, типах данных и рефлексивные `owl:sameAs`.
Общее число добавленных троек нельзя интерпретировать как число новых независимых фактов о библиотеках.


In [8]:
inferred = clone_graph(kg)
for triple in ontology:
    inferred.add(triple)
asserted = set(inferred)

owlrl.DeductiveClosure(
    owlrl.OWLRL_Semantics, axiomatic_triples=False, datatype_axioms=False
).expand(inferred)

added = set(inferred) - asserted
assert asserted <= set(inferred), 'Вывод должен только добавлять тройки'
assert len(inferred) == len(asserted) + len(added)
assert added and not (added & asserted)

delta_graph = Graph()
for prefix, namespace in inferred.namespaces():
    delta_graph.bind(prefix, namespace)
for triple in added:
    delta_graph.add(triple)
delta_path = artifacts / 'owlrl_added_triples.ttl'
delta_graph.serialize(destination=delta_path, format='turtle')
assert isomorphic(delta_graph, Graph().parse(delta_path, format='turtle'))

reasoning_stats = {
    'Троек данных': len(kg),
    'Троек онтологии': len(ontology),
    'До вывода (объединение)': len(asserted),
    'После вывода': len(inferred),
    'Добавлено троек': len(added),
    'Из них рефлексивных owl:sameAs': sum(p == OWL.sameAs and s == o for s, p, o in added),
}
display(pd.DataFrame(reasoning_stats.items(), columns=['Метрика', 'Значение']))
print('Все добавленные тройки сохранены:', delta_path.relative_to(project).as_posix())
(artifacts / 'owlrl_summary.json').write_text(
    json.dumps(reasoning_stats, ensure_ascii=False, indent=2), encoding='utf-8'
)

predicate_counts = Counter(p for s, p, o in added)
display(pd.DataFrame([
    {'Предикат добавленной тройки': short(p), 'Количество': count}
    for p, count in sorted(predicate_counts.items(), key=lambda item: (-item[1], str(item[0])))
]))


,Метрика,Значение
0,Троек данных,9302
1,Троек онтологии,155
2,До вывода (объединение),9457
3,После вывода,22381
4,Добавлено троек,12924
5,Из них рефлексивных owl:sameAs,3935


Все добавленные тройки сохранены: data/tasks_5_6/owlrl_added_triples.ttl


,Предикат добавленной тройки,Количество
0,rdf:type,6025
1,owl:sameAs,3992
2,pte:hasVersion,1719
3,<http://www.wikidata.org/prop/direct/P1547>,208
4,pte:dependsOnSoftware,156
5,<http://www.wikidata.org/prop/direct/P1324>,106
6,<http://www.wikidata.org/prop/direct/P275>,106
7,rdfs:subClassOf,71
8,pte:hasLicense,53
9,pte:hasPrimaryLicense,53


### 6.6. Какие именно тройки добавились и почему

Ниже показаны примеры четырёх механизмов вывода с исходными предпосылками:

* **Иерархия классов:** экземпляр `ProbabilisticAndGraphLibrary` становится экземпляром `MachineLearningLibrary`, а затем `SoftwareLibrary`.
* **Эквивалентность классов:** принадлежность локальному `SoftwareLibrary` переносится на эквивалентный класс `wd:Q188860`.
* **Эквивалентность свойств:** связь `dependsOnSoftware` порождает связь `wdt:P1547`.
* **Тождество сущностей:** свойства локальной библиотеки переносятся на её QID через `owl:sameAs`.

Для каждого примера программно проверяется, что тройка отсутствовала до вывода и присутствует в сохранённой разности.
Эквивалентности взяты из онтологии задания 3; корректность их предметного смысла сам механизм вывода не проверяет.
Вывод выполняется локально и не загружает дополнительные сведения из Wikidata.


In [9]:
name = kg.value(library, PTE.libraryName)
wd_library = kg.value(library, OWL.sameAs)
examples = [
    ('Иерархия классов', (library, RDF.type, PTE.MachineLearningLibrary), [
        (library, RDF.type, PTE.ProbabilisticAndGraphLibrary),
        (PTE.ProbabilisticAndGraphLibrary, RDFS.subClassOf, PTE.MachineLearningLibrary),
    ]),
    ('Транзитивная иерархия', (library, RDF.type, PTE.SoftwareLibrary), [
        (library, RDF.type, PTE.MachineLearningLibrary),
        (PTE.MachineLearningLibrary, RDFS.subClassOf, PTE.SoftwareLibrary),
    ]),
    ('Эквивалентность классов', (library, RDF.type, WD.Q188860), [
        (library, RDF.type, PTE.SoftwareLibrary),
        (PTE.SoftwareLibrary, OWL.equivalentClass, WD.Q188860),
    ]),
    ('Эквивалентность свойств', (library, WDT.P1547, PTE.PyTorch), [
        (library, PTE.dependsOnSoftware, PTE.PyTorch),
        (PTE.dependsOnSoftware, OWL.equivalentProperty, WDT.P1547),
    ]),
    ('Тождество owl:sameAs', (wd_library, PTE.libraryName, name), [
        (library, OWL.sameAs, wd_library),
        (library, PTE.libraryName, name),
    ]),
]

def triple_text(triple):
    return ' '.join(short(term) for term in triple) + ' .'

example_rows = []
for mechanism, triple, premises in examples:
    assert triple not in asserted and triple in added
    assert all(premise in inferred for premise in premises)
    example_rows.append({
        'Механизм': mechanism,
        'Добавленная тройка': triple_text(triple),
        'Предпосылки': '\n'.join(
            ('[исходная] ' if premise in asserted else '[выведенная] ') + triple_text(premise)
            for premise in premises
        ),
    })
display(pd.DataFrame(example_rows))
(artifacts / 'owlrl_examples.json').write_text(
    json.dumps(example_rows, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('Первые 15 новых троек о локальных библиотеках (детерминированная сортировка):')
library_triples = sorted(
    (triple for triple in added if triple[0] in all_libraries and triple[1] != OWL.sameAs),
    key=lambda triple: tuple(term.n3() for term in triple),
)
for triple in library_triples[:15]:
    print(triple_text(triple))


,Механизм,Добавленная тройка,Предпосылки
0,Иерархия классов,pte:Lib_PyKEEN_Q96631404 rdf:type pte:MachineLearningLibrary .,[исходная] pte:Lib_PyKEEN_Q96631404 rdf:type pte:ProbabilisticAndGraphLibrary .\n[исходная] pte:ProbabilisticAndGraphLibrary rdfs:subClassOf pte:MachineLearningLibrary .
1,Транзитивная иерархия,pte:Lib_PyKEEN_Q96631404 rdf:type pte:SoftwareLibrary .,[выведенная] pte:Lib_PyKEEN_Q96631404 rdf:type pte:MachineLearningLibrary .\n[исходная] pte:MachineLearningLibrary rdfs:subClassOf pte:SoftwareLibrary .
2,Эквивалентность классов,pte:Lib_PyKEEN_Q96631404 rdf:type wd:Q188860 .,[выведенная] pte:Lib_PyKEEN_Q96631404 rdf:type pte:SoftwareLibrary .\n[исходная] pte:SoftwareLibrary owl:equivalentClass wd:Q188860 .
3,Эквивалентность свойств,pte:Lib_PyKEEN_Q96631404 <http://www.wikidata.org/prop/direct/P1547> pte:PyTorch .,[исходная] pte:Lib_PyKEEN_Q96631404 pte:dependsOnSoftware pte:PyTorch .\n[исходная] pte:dependsOnSoftware owl:equivalentProperty <http://www.wikidata.org/prop/direct/P1547> .
4,Тождество owl:sameAs,"wd:Q96631404 pte:libraryName ""PyKEEN""^^xsd:string .","[исходная] pte:Lib_PyKEEN_Q96631404 owl:sameAs wd:Q96631404 .\n[исходная] pte:Lib_PyKEEN_Q96631404 pte:libraryName ""PyKEEN""^^xsd:string ."


Первые 15 новых троек о локальных библиотеках (детерминированная сортировка):
pte:Lib_PyKEEN_Q96631404 pte:dependsOnSoftware wd:Q47509047 .
pte:Lib_PyKEEN_Q96631404 rdf:type pte:MachineLearningLibrary .
pte:Lib_PyKEEN_Q96631404 rdf:type pte:SoftwareLibrary .
pte:Lib_PyKEEN_Q96631404 rdf:type owl:Thing .
pte:Lib_PyKEEN_Q96631404 rdf:type wd:Q188860 .
pte:Lib_PyKEEN_Q96631404 rdf:type wd:Q21169670 .
pte:Lib_PyKEEN_Q96631404 <http://www.wikidata.org/prop/direct/P1324> pte:Repo_PyKEEN_Q96631404 .
pte:Lib_PyKEEN_Q96631404 <http://www.wikidata.org/prop/direct/P1547> pte:PyTorch .
pte:Lib_PyKEEN_Q96631404 <http://www.wikidata.org/prop/direct/P1547> wd:Q47509047 .
pte:Lib_PyKEEN_Q96631404 <http://www.wikidata.org/prop/direct/P275> pte:License_лицензия_MIT .
pte:Lib_PyTorch_Lightning_Q85847011 pte:dependsOnSoftware wd:Q47509047 .
pte:Lib_PyTorch_Lightning_Q85847011 rdf:type pte:MachineLearningLibrary .
pte:Lib_PyTorch_Lightning_Q85847011 rdf:type pte:SoftwareLibrary .
pte:Lib_PyTorch_Lightning_

## Итоги выполнения заданий 5 и 6

1. **Задание 5:** собран JSONL из сохранённых README для всех 52 сущностей выборки; для каждой приведена ссылка на источник. Проверены уникальность QID, непустота текстов, покрытие выборки и обратное чтение файла.
2. **Задание 6, проверка:** применены пять видов ограничений SHACL. На исходном графе получен полный отчёт pyshacl. В отдельной экспериментальной копии выявлены пять заранее внесённых ошибок; по каждому типу объяснено исправление, после которого проверка повторена.
3. **Задание 6, вывод:** выполнен OWL 2 RL, сохранено полное множество добавленных троек; приведены примеры с предпосылками и проверкой отсутствия этих троек в исходном графе.

Основные файлы результатов:

| Файл в `data/tasks_5_6/` | Содержание |
| :--- | :--- |
| `corpus.jsonl`, `corpus_sources.csv` | Корпус и полный список источников |
| `shapes.shacl.ttl` | Формы, использованные в проверке |
| `shacl_original.txt`, `shacl_original.ttl` | Полный результат pyshacl для исходных данных |
| `validation_demo_invalid.ttl` | Экспериментальная копия с искусственными ошибками |
| `shacl_demo_before.*`, `shacl_demo_after.*` | Полные результаты pyshacl до и после исправления копии |
| `validation_summary.csv` | Сводка трёх проверок |
| `owlrl_added_triples.ttl` | Все добавленные тройки |
| `owlrl_summary.json`, `owlrl_examples.json` | Статистика вывода и примеры с объяснениями |
| `environment.json` | Версии Python и библиотек |

Исходные граф, онтология и корпус предыдущих заданий не изменяются.


In [10]:
assert len(loaded_corpus) == len(entity_ids) == 52
assert base_conforms and not demo_conforms and fixed_conforms
assert len(expected_components) == 5
assert set(kg) == kg_before_validation
assert set(repaired) == set(kg)
assert set(inferred) == asserted | added

print(f'Задание 5: {len(loaded_corpus)}/{len(entity_ids)} сущностей с текстом и источником; JSONL сохранён.')
print('Задание 6: исходный граф — Conforms: True; эксперимент — 5 нарушений; после исправления — 0.')
print(f'OWL 2 RL: {len(asserted):,} → {len(inferred):,} троек; добавлено {len(added):,}.')
print('Все итоговые проверки пройдены.')


Задание 5: 52/52 сущностей с текстом и источником; JSONL сохранён.


Задание 6: исходный граф — Conforms: True; эксперимент — 5 нарушений; после исправления — 0.
OWL 2 RL: 9,457 → 22,381 троек; добавлено 12,924.
Все итоговые проверки пройдены.
